# ScorePredictor — Preprocessing Walkthrough (Colab-style)

This notebook runs the **exact same pipeline** as `train_model.py`, but one step per
cell so you can *see* every stage:

1. Generate a **raw, messy** dataset (missing values, duplicates, outliers)
2. Inspect the raw data
3. Find & handle **missing values** (mean imputation)
4. Find & remove **duplicate rows**
5. Find & remove **outliers**
6. **Scale** the features (StandardScaler)
7. Train / test split
8. Train the model & evaluate (R²)

**How to use:** open in Google Colab or Jupyter and choose *Run all*. Each cell prints
its output right below it (tables, counts, charts), just like Colab.

In [ ]:
import numpy as np
import pandas as pd
# matplotlib is optional (Colab has it built in). If it's missing locally, run
# `pip install matplotlib` -- but every DATA step still works without charts.
try:
    import matplotlib.pyplot as plt
    HAS_PLT = True
except ModuleNotFoundError:
    HAS_PLT = False
    print('matplotlib not installed -> charts will be skipped (data steps still run).')
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

# Same fixed seed as train_model.py -> identical numbers (655 raw -> 635 cleaned).
RNG = np.random.default_rng(42)
N_STUDENTS = 640
pd.set_option('display.max_columns', None)

## Step 1 — Generate the RAW (messy) dataset

Real student records are private, so we synthesize data and **deliberately inject**
the same problems real data collection produces: blank cells, duplicate rows, and
data-entry typos (e.g. `250` study hours instead of `25.0`).

In [ ]:
def generate_raw_dataset(n=N_STUDENTS):
    study_hours = np.clip(RNG.normal(14.2, 5.0, n), 2, 30)
    attendance = np.clip(RNG.normal(81, 10, n), 50, 100)
    noise = RNG.normal(0, 3.4, n)
    exam_score = np.clip(11 + 2.1 * study_hours + 0.45 * attendance + noise, 0, 100)
    df = pd.DataFrame({
        'study_hours': np.round(study_hours, 1),
        'attendance': np.round(attendance, 1),
        'exam_score': np.round(exam_score, 1),
    })
    # Inject MISSING VALUES (~3% blank cells in two columns)
    n_missing = int(0.03 * n)
    df.loc[RNG.choice(df.index, n_missing, replace=False), 'study_hours'] = np.nan
    df.loc[RNG.choice(df.index, n_missing, replace=False), 'attendance'] = np.nan
    # Inject DUPLICATE ROWS (same record entered twice)
    df = pd.concat([df, df.sample(n=15, random_state=1)], ignore_index=True)
    # Inject OUTLIERS (impossible study hours = typos)
    df.loc[RNG.choice(df.index, 5, replace=False), 'study_hours'] = [250.0, 300.0, 180.0, 220.0, 275.0]
    # Shuffle so the problems are spread throughout
    return df.sample(frac=1, random_state=7).reset_index(drop=True)

raw = generate_raw_dataset()
print('Raw shape (rows, columns):', raw.shape)
raw.head(10)

## Step 2 — Inspect the raw data

`describe()` already hints at problems: look at the **max** of `study_hours` (it should
be ~30 but shows 300 — outliers) and the **count** row (less than the total — missing values).

In [ ]:
raw.describe().round(2)

In [ ]:
raw.info()

## Step 3a — Find the MISSING values

`isnull().sum()` counts blanks per column. Then we look at a few actual rows that
contain a blank (shown as `NaN`).

In [ ]:
print('Missing values per column:')
print(raw.isnull().sum())
print('\nTotal missing cells:', int(raw.isnull().sum().sum()))

In [ ]:
# Show some rows that actually have a missing value
raw[raw.isnull().any(axis=1)].head(10)

In [ ]:
# Visualize where the missing cells are (bright streaks = missing)
if HAS_PLT:
    plt.figure(figsize=(8, 3))
    plt.imshow(raw.isnull(), aspect='auto', cmap='viridis', interpolation='nearest')
    plt.xticks(range(len(raw.columns)), raw.columns)
    plt.title('Missing-value map (bright = missing)')
    plt.xlabel('column'); plt.ylabel('row index')
    plt.tight_layout(); plt.show()
else:
    print('(chart skipped - matplotlib not installed)')

### Handle missing values — mean imputation

We fill each blank with that column's **mean** (a simple, common strategy). After this,
the missing count is **0**.

In [ ]:
df = raw.copy()
df = df.fillna(df.mean(numeric_only=True))
print('Missing values AFTER imputation:', int(df.isnull().sum().sum()))

## Step 3b — Find & remove DUPLICATE rows

`duplicated()` flags rows that are exact copies of an earlier row.

In [ ]:
print('Duplicate rows found:', int(df.duplicated().sum()))
# Show a few duplicated rows (keep=False shows both the original and the copy)
df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(10)

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Removed {before - len(df)} duplicate row(s). Rows now: {len(df)}')

## Step 3c — Find & remove OUTLIERS

Valid weekly study hours are ~0–40. Anything above that is a data-entry error
(e.g. `250` meant `25.0`). The boxplot makes the outliers obvious.

In [ ]:
# The impossible values
df[df['study_hours'] > 40]

In [ ]:
if HAS_PLT:
    plt.figure(figsize=(6, 2.5))
    plt.boxplot(df['study_hours'], vert=False)
    plt.title('study_hours BEFORE removing outliers')
    plt.xlabel('study hours'); plt.tight_layout(); plt.show()
else:
    print('(boxplot skipped - matplotlib not installed)')

In [ ]:
before = len(df)
df = df[(df['study_hours'] >= 0) & (df['study_hours'] <= 40)].reset_index(drop=True)
print(f'Removed {before - len(df)} outlier row(s). Clean rows: {len(df)}')

if HAS_PLT:
    plt.figure(figsize=(6, 2.5))
    plt.boxplot(df['study_hours'], vert=False)
    plt.title('study_hours AFTER removing outliers')
    plt.xlabel('study hours'); plt.tight_layout(); plt.show()
else:
    print('(boxplot skipped - matplotlib not installed)')

## Step 4 — Feature scaling (StandardScaler)

`study_hours` (~2–30) and `attendance` (~50–100) live on different scales. StandardScaler
rescales each to **mean 0, std 1** so neither feature dominates. Compare the `mean`/`std`
rows before vs after.

In [ ]:
X = df[['study_hours', 'attendance']]
y = df['exam_score']

print('BEFORE scaling:')
print(X.describe().round(2).loc[['mean', 'std', 'min', 'max']])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print('\nAFTER scaling (mean ~0, std ~1):')
print(X_scaled_df.describe().round(2).loc[['mean', 'std', 'min', 'max']])

## Step 5 — Train / test split (80 / 20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)
print(f'Training set: {X_train.shape[0]} rows (80%)')
print(f'Testing set : {X_test.shape[0]} rows (20%)')

## Step 6 — Train the model & Step 7 — Evaluate

R² ≈ **0.89** means the model explains ~89% of the variation in exam scores.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f'Coefficients (scaled): study_hours={model.coef_[0]:.3f}, attendance={model.coef_[1]:.3f}')
print(f'Intercept            : {model.intercept_:.3f}')
print(f'R-squared            : {r2:.3f}  (~{r2*100:.0f}% of variance explained)')
print(f'MAE                  : {mae:.2f} points')

In [ ]:
# Predicted vs actual — points near the diagonal = good predictions
if HAS_PLT:
    plt.figure(figsize=(5, 5))
    plt.scatter(y_test, y_pred, alpha=0.5)
    lims = [40, 100]
    plt.plot(lims, lims, 'r--')
    plt.xlabel('Actual exam score'); plt.ylabel('Predicted exam score')
    plt.title(f'Predicted vs Actual (R² = {r2:.2f})')
    plt.tight_layout(); plt.show()
else:
    print('(scatter plot skipped - matplotlib not installed)')

## (Optional) Step 8 — Save the artifacts

Uncomment to write `model.pkl`, `scaler.pkl`, and the cleaned CSV — exactly what
`train_model.py` produces for the Flask backend.

In [ ]:
# import pickle
# with open('model.pkl', 'wb') as f: pickle.dump(model, f)
# with open('scaler.pkl', 'wb') as f: pickle.dump(scaler, f)
# df.to_csv('student_data.csv', index=False)
# print('Saved model.pkl, scaler.pkl, student_data.csv')